1) Создать качественный датасет для учителя-LLM
2) Создать качественный датасет для ученика-LLM
3) Обучить ученика-LLM

20К - для обучения
5K - для тестов

In [1]:
!pip install sentence_transformers
!pip install langchain_text_splitters
!pip install transformers torch accelerate bitsandbytes
!pip install faiss-gpu-cu12 rank_bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 50.4 MB/s eta 0:00:00


In [2]:
!git clone https://github.com/chipslays/russian-recipes-parser.git

Cloning into 'russian-recipes-parser'...
remote: Enumerating objects: 14584, done.
remote: Counting objects: 100% (14584/14584), done.
remote: Compressing objects: 100% (1592/1592), done.
remote: Total 14584 (delta 12973), reused 14580 (delta 12971), pack-reused 0 (from 0)
Receiving objects: 100% (14584/14584), 18.55 MiB | 16.84 MiB/s, done.
Resolving deltas: 100% (12973/12973), done.


In [89]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from langchain_text_splitters import RecursiveCharacterTextSplitter
import faiss
import numpy as np
from rank_bm25 import BM25Okapi
import re

import json
import glob
import os
import gc
from typing import Literal, Optional

In [5]:
# Путь к папке с рецептами (после клонирования репозитория)
recipes_path = "russian-recipes-parser/storage/recipes/[1-3][0-9][0-9]/*.json"

parsed_recipe_list = []

for filepath in glob.glob(recipes_path):
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            recipe_data = json.load(f)
        parsed_recipe = {
            "title": recipe_data.get('title', ''),
            "ingredients": recipe_data.get('ingredients', ''),
            "instruction": recipe_data.get('instruction', ''),
            "full_text": "",
        }

        # Собираем полный текст рецепта из полей JSON
        full_recipe_text = f"{recipe_data.get('title', '')}\n\n"

        ingredient_list = recipe_data.get('ingredients', '')[0]
        full_recipe_text += f"{ingredient_list['name']}:\n"
        for index, ingredient in enumerate(ingredient_list['list']):
          full_recipe_text += f"{index + 1}) {ingredient['name']} {ingredient['value'] or ''} {ingredient['type'] or ''}\n"

        full_recipe_text += f"\nПриготовление:\n" # Здесь лежат шаги
        for index, step in enumerate(recipe_data.get('instruction', '')):
          full_recipe_text += f"{index + 1}) {step['text']}\n"
        parsed_recipe["full_text"] = full_recipe_text
        parsed_recipe_list.append(parsed_recipe)

    except Exception as e:
        print(f"Ошибка при обработке файла {filepath}: {e}")

print(f"Загружено {len(parsed_recipe_list)} рецептов.")
# Посмотрим пример
print("\n--- Пример рецепта ---")
print(parsed_recipe_list[0]["full_text"])

Загружено 7653 рецептов.

--- Пример рецепта ---
Майонез «Провансаль» на скорую руку

Ингредиенты:
1) масло растительное 200 мл
2) яйца 2 шт.
3) горчица 1 ч.л.
4) соль 1/2 ч.л.
5) перец черный молотый  
6) сахар  

Приготовление:
1) Желтки взбить, добавить сахар, соль, черный молотый перец и готовую горчицу и снова взбить. Вливать порциями растительное рафинированное масло, продолжать взбивать. Добавить уксус, отделенный белок и снова взбить. Можно добавить еще соль и уксус по вкусу, немного сметаны.



In [6]:
class RecipeChunker:
    """Создает чанки разного масштаба для одного рецепта"""

    def __init__(self, model_name="deepvk/USER2-small", truncate_dim=128):
        self.model = SentenceTransformer(model_name, truncate_dim=truncate_dim)

    def chunk_recipe(self, parsed_recipe, max_chunk_size=512):
        chunks = []

        # Уровень 1: Целый рецепт (глобальный контекст)
        # chunks.append({
        #     'text': parsed_recipe.get('full_text', ''),
        #     'scale': 'full',
        #     'recipe_name': parsed_recipe.get('name', '')
        # })

        # Уровень 2: Секции (ингредиенты, шаги)
        if parsed_recipe['ingredients']:
          ingredient_list = parsed_recipe['ingredients'][0]
          ing_text = f"Ингредиенты для рецепта \"{parsed_recipe.get('name', '')}\":\n"
          for index, ingredient in enumerate(ingredient_list['list']):
            ing_text += f"{index + 1}) {ingredient['name']} {ingredient['value'] or ''} {ingredient['type'] or ''}\n"
          chunks.append({
            'text': ing_text,
            'scale': 'section',
            'type': 'ingredients',
            'recipe_name': parsed_recipe.get('name', '')
          })

        # Уровень 3: Отдельные шаги (детали)
        # Не нравится везде префикс
        if parsed_recipe['instruction']:
          splitter = RecursiveCharacterTextSplitter(
              chunk_size=max_chunk_size,
              chunk_overlap=30,
              separators=[". ", ", ", " "]
          )
          for index, step in enumerate(parsed_recipe['instruction']):
            step_text = f"Инструкции для рецепта \"{parsed_recipe.get('name', '')}\":\n"
            step_text += f"{index + 1}) {step['text']}\n"
            if len(step_text) <= max_chunk_size:
              chunks.append({
                'text': step_text,
                'scale': 'step',
                'step_num': index + 1,
                'recipe_name': parsed_recipe.get('name', '')
              })
              continue
            sub_chunks = splitter.split_text(step_text)
            for part, sub in enumerate(sub_chunks):
              chunks.append({
                'text': sub,
                'type': 'step_part',
                'step': index + 1,
                'part': part + 1,
                'recipe': parsed_recipe.get('name', '')
              })

        return chunks

In [7]:
# Использование
chunker = RecipeChunker()
chunks = []
for parsed_recipe in parsed_recipe_list:
  chunks.extend(chunker.chunk_recipe(parsed_recipe))

print(f"Создано {len(chunks)} чанков разного масштаба")
for i in range(3):
  chunk = chunks[i]
  print(f"- {chunk['scale']}: {chunk['text']}...")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/359 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/837 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Создано 39225 чанков разного масштаба
- section: Ингредиенты для рецепта "":
1) масло растительное 200 мл
2) яйца 2 шт.
3) горчица 1 ч.л.
4) соль 1/2 ч.л.
5) перец черный молотый  
6) сахар  
...
- step: Инструкции для рецепта "":
1) Желтки взбить, добавить сахар, соль, черный молотый перец и готовую горчицу и снова взбить. Вливать порциями растительное рафинированное масло, продолжать взбивать. Добавить уксус, отделенный белок и снова взбить. Можно добавить еще соль и уксус по вкусу, немного сметаны.
...
- section: Ингредиенты для рецепта "":
1) фарш 100 г
2) рис 100 г
3) картофель 7-8 шт.
4) морковь 2 шт.
5) перец сладкий болгарский 1 шт.
6) томатная паста 2 ст.л.
7) лук репчатый 3 головки
8) чеснок 3 зубчика
9) перец черный молотый  
10) масло растительное  
11) лавровый лист  
12) соль  
...


In [8]:
# genrate query

# # 4-bit квантизация для экономии памяти
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16
# )

# model_name = "Qwen/Qwen2.5-7B-Instruct"
# tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=os.environ["HF_TOKEN"])
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     quantization_config=bnb_config,
#     device_map="auto",
#     trust_remote_code=True
# )

# def generate_questions(chunk_text, num_questions=3):
#     prompt = f"""Ты — помощник, который генерирует вопросы к рецептам.
# На основе текста рецепта придумай {num_questions} вопроса, на которые отвечает этот текст.

# Текст рецепта: {chunk_text}

# Вопросы (каждый с новой строки, без нумерации):"""

#     inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
#     outputs = model.generate(
#         **inputs,
#         max_new_tokens=200,
#         temperature=0.8,
#         do_sample=True
#     )

#     response = tokenizer.decode(outputs[0], skip_special_tokens=True)
#     questions = response.replace(prompt, '').strip().split('\n')
#     return [q.strip() for q in questions if q.strip() and '?' in q][:num_questions]


def generate_questions(chunk_text, question_length=150, num_questions=3):
  questions = []

  for i in range(0, len(chunk_text), question_length):
    question = "".join(chunk_text[i:i+question_length])
    questions.append(question)
  return questions

In [9]:
questions = []
for i in range(3):
  questions.extend(generate_questions(chunks[i]['text']))

for item in questions:
  print(item)

Ингредиенты для рецепта "":
1) масло растительное 200 мл
2) яйца 2 шт.
3) горчица 1 ч.л.
4) соль 1/2 ч.л.
5) перец черный молотый  
6) сахар  

Инструкции для рецепта "":
1) Желтки взбить, добавить сахар, соль, черный молотый перец и готовую горчицу и снова взбить. Вливать порциями растительно
е рафинированное масло, продолжать взбивать. Добавить уксус, отделенный белок и снова взбить. Можно добавить еще соль и уксус по вкусу, немного сметан
ы.

Ингредиенты для рецепта "":
1) фарш 100 г
2) рис 100 г
3) картофель 7-8 шт.
4) морковь 2 шт.
5) перец сладкий болгарский 1 шт.
6) томатная паста 2 ст.
л.
7) лук репчатый 3 головки
8) чеснок 3 зубчика
9) перец черный молотый  
10) масло растительное  
11) лавровый лист  
12) соль  



In [10]:
class BM25Teacher:
    def __init__(self, chunks):
        print("Инициализация BM25Teacher...")

        # Извлекаем текст, если chunks — список словарей
        if isinstance(chunks[0], dict):
            self.chunks = chunks
            texts = [chunk['text'] for chunk in chunks]
        else:
            self.chunks = chunks
            texts = chunks

        # Простая токенизация для BM25
        print("Токенизация для BM25...")
        tokenized = []
        for text in texts:
            # Удаляем пунктуацию и приводим к нижнему регистру
            words = re.findall(r'\w+', text.lower())
            tokenized.append(words)

        self.bm25 = BM25Okapi(tokenized)
        print(f"BM25Teacher готов. Индекс содержит {len(tokenized)} документов")

    def search(self, question, k=100):
        """Возвращает топ-k чанков"""
        # Токенизируем запрос
        tokenized_query = re.findall(r'\w+', question.lower())

        # Получаем оценки BM25
        scores = self.bm25.get_scores(tokenized_query)

        # Получаем индексы топ-k (от наибольшего к наименьшему)
        indices = np.argsort(scores)[-k:][::-1]

        return indices, scores[indices]

In [11]:
class DenseTeacher:
    def __init__(self, chunks, model_name="deepvk/USER2-small"):
        print("Инициализация DenseTeacher...")
        self.model = SentenceTransformer(model_name)

        # Извлекаем текст, если chunks — список словарей
        if isinstance(chunks[0], dict):
            self.chunks = chunks
            texts = [chunk['text'] for chunk in chunks]
        else:
            self.chunks = chunks
            texts = chunks

        # Индексация всех чанков
        print("Кодируем чанки для dense поиска...")
        self.embeddings = self.model.encode(
            texts,
            prompt_name="search_document",
            show_progress_bar=True,
            batch_size=32
        )
        faiss.normalize_L2(self.embeddings)

        # FAISS индекс
        self.index = faiss.IndexFlatIP(self.embeddings.shape[1])
        self.index.add(self.embeddings.astype('float32'))
        print(f"DenseTeacher готов. Индекс содержит {self.index.ntotal} векторов")

    def search(self, question, k=100):
        """Возвращает топ-k чанков"""
        q_emb = self.model.encode([question], prompt_name="search_query")
        faiss.normalize_L2(q_emb)
        scores, indices = self.index.search(q_emb.astype('float32'), k)
        return indices[0], scores[0]

In [12]:
class MultiTeacherRetriever:
    def __init__(self, chunks):
        self.chunks = chunks
        self.teachers = {}

    def add_teacher(self, name, teacher):
        self.teachers[name] = teacher
        print(f"Добавлен учитель: {name}")

    def get_rankings(self, question, k=100):
        """Получает списки от всех учителей"""
        rankings = {}
        for name, teacher in self.teachers.items():
            print(f"Поиск у учителя {name}...")
            indices, scores = teacher.search(question, k)
            rankings[name] = {
                'indices': indices.tolist() if hasattr(indices, 'tolist') else list(indices),
                'scores': scores.tolist() if hasattr(scores, 'tolist') else list(scores)
            }
        return rankings

    def fuse_rankings(self, rankings, weights=None):
        """Объединяет списки через Reciprocal Rank Fusion"""
        if weights is None:
            weights = {name: 1.0 for name in rankings}

        # RRF fusion
        rrf_scores = {}
        k_constant = 60  # стандартное значение для RRF

        for teacher_name, rank_data in rankings.items():
            weight = weights.get(teacher_name, 1.0)
            for rank, idx in enumerate(rank_data['indices']):
                if idx not in rrf_scores:
                    rrf_scores[idx] = 0
                # RRF формула: weight / (k + rank)
                rrf_scores[idx] += weight * (1.0 / (k_constant + rank))

        # Сортируем по убыванию
        sorted_indices = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
        return [idx for idx, score in sorted_indices]

# Создаем ретривер
retriever = MultiTeacherRetriever(chunks)

# Добавляем учителей
retriever.add_teacher('dense', DenseTeacher(chunks))
retriever.add_teacher('bm25', BM25Teacher(chunks))

# Тестируем поиск
question = "Как сварить борщ со свеклой?"
print(f"\nЗапрос: {question}")

rankings = retriever.get_rankings(question, k=3)

print("\nРезультаты отдельных учителей:")
for name, rank_data in rankings.items():
    print(f"\n{name.upper()}:")
    for i, (idx, score) in enumerate(zip(rank_data['indices'], rank_data['scores'])):
        print(f"  {i+1}. Чанк {idx}: {chunks[idx]['text'][:150]}... (score: {score:.4f})")

# Объединяем результаты
fused_list = retriever.fuse_rankings(rankings, weights={'dense': 0.5, 'bm25': 0.5})

print("\nFused результат (объединенный):")
for i, idx in enumerate(fused_list[:3]):
    print(f"  {i+1}. Чанк {idx}: {chunks[idx]['text'][:150]}...")

Инициализация DenseTeacher...


Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

Кодируем чанки для dense поиска...


Batches:   0%|          | 0/1226 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:321: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(
W0318 14:06:15.767000 8200 torch/_inductor/utils.py:1679] [1/0_1] Not enough SMs to use max_autotune_gemm mode


DenseTeacher готов. Индекс содержит 39225 векторов
Добавлен учитель: dense
Инициализация BM25Teacher...
Токенизация для BM25...
BM25Teacher готов. Индекс содержит 39225 документов
Добавлен учитель: bm25

Запрос: Как сварить борщ со свеклой?
Поиск у учителя dense...
Поиск у учителя bm25...

Результаты отдельных учителей:

DENSE:
  1. Чанк 23042: Инструкции для рецепта "":
4) Обжаривать, пока зажарка не будет одного цвета, в конце положить томатную пасту, размешать, обжаривать еще шестьдесят се... (score: 0.8927)
  2. Чанк 23043: Инструкции для рецепта "":
5) Шинковать капусту, когда борщ приобретет свекольного цвета, положить капусту в кастрюльку, варить пятнадцать минут, поло... (score: 0.8920)
  3. Чанк 24762: Инструкции для рецепта "":
2) Морковь, петрушку и лук нарезать соломкой, спассеровать на столовом маргарине. В кипящий бульон положить сырой нарезанны... (score: 0.8912)

BM25:
  1. Чанк 26650: Инструкции для рецепта "":
8) Обжарку добавить в картошку с мясом вместе со свеклой.


In [13]:
import random


triplet_list = []
for question_item in questions:
  rankings = retriever.get_rankings(question_item, k=50)
  fused_list = retriever.fuse_rankings(rankings, weights={'dense': 0.5, 'bm25': 0.5})
  positive_idx = random.choice(fused_list[:10])
  # Рандомно выбираем hard negative из 46-50 (индексы 45-49)
  hard_negative_idx = random.choice(fused_list[45:50])  # 45,46,47,48,49
  # Получаем текст чанков
  positive_chunk = chunks[positive_idx]
  hard_negative_chunk = chunks[hard_negative_idx]
  triplet_list.append((question_item, positive_chunk['text'], hard_negative_chunk['text']))

for i in range(5):
  print(triplet_list[i])


Поиск у учителя dense...
Поиск у учителя bm25...
Поиск у учителя dense...
Поиск у учителя bm25...
Поиск у учителя dense...
Поиск у учителя bm25...
Поиск у учителя dense...
Поиск у учителя bm25...
Поиск у учителя dense...
Поиск у учителя bm25...
Поиск у учителя dense...
Поиск у учителя bm25...
('Ингредиенты для рецепта "":\n1) масло растительное 200 мл\n2) яйца 2 шт.\n3) горчица 1 ч.л.\n4) соль 1/2 ч.л.\n5) перец черный молотый  \n6) сахар  \n', 'Ингредиенты для рецепта "":\n1) масло растительное 200 мл\n2) молоко 150 мл\n3) масло оливковое 100 мл\n4) сок лимонный/уксус 2-3 ст.л.\n5) горчица 1 ст.л.\n6) соль 1 ч.л.\n', 'Ингредиенты для рецепта "":\n1) кабачки 2 шт.\n2) яйца куриные 3 шт.\n3) мука рисовая 3 ст.л.\n4) чеснок 3 зубчика\n5) соль ½ ч.л.\n6) перец черный молотый ½ ч.л.\n7) укроп  \n8) масло растительное 100 мл\n')
('Инструкции для рецепта "":\n1) Желтки взбить, добавить сахар, соль, черный молотый перец и готовую горчицу и снова взбить. Вливать порциями растительно', 'Инструк

In [104]:
class PovaryoshkaRetriever(nn.Module):
    def __init__(
      self,
      encoder_name="deepvk/USER2-small",
      temperature=0.05
    ):
      super().__init__()
      self.temperature = temperature
      self.encoder = SentenceTransformer(encoder_name)
      self.document_list: Optional[list[str]] = None
      self.retriever_index = None


    def forward(
      self,
      query_list: list[str],
      positive_doc_list: list[str],
      hard_negative_doc_list: list[str]
    ) -> torch.Tensor:
      query_embedding_list = self._encode(
        query_list,
        is_query=True,
      )
      positive_doc_list.extend(hard_negative_doc_list)
      doc_embedding_list = self._encode(positive_doc_list)

      similarity_with_temperature = torch.matmul(query_embedding_list, doc_embedding_list.T) / self.temperature
      batch_size = query_embedding_list.size(0)
      positive_similarity_with_temperature = similarity_with_temperature[:, :batch_size].diag()
      log_sum_exp = torch.logsumexp(similarity_with_temperature, dim=1)
      loss = -(positive_similarity_with_temperature - log_sum_exp)
      return loss.mean()


    def _encode(
      self,
      text_list: list[str],
      is_query=False,
      is_normalize=True,
      device: Optional[Literal["cuda", "cpu"]]=None
    ) -> torch.Tensor:
      if is_query:
        text_list = [f"query:  {text}" for text in text_list]
      else:
        text_list = [f"passage: {text}" for text in text_list]

      retriever_device = next(self.parameters()).device
      specified_device = device or retriever_device
      self.encoder.to(specified_device)
      tokenized_text_list = self.encoder.tokenize(text_list)
      tokenized_text_list = {key: value.to(specified_device) for key, value in tokenized_text_list.items()}
      encoder_output: dict[str, torch.Tensor] = self.encoder(tokenized_text_list)
      embedding_text_list = encoder_output["sentence_embedding"]

      if is_normalize:
        embedding_text_list = F.normalize(embedding_text_list, dim=1)

      return embedding_text_list


    def encode_queries(
      self,
      query_list: list[str],
      is_normalize=True
    ) -> torch.Tensor:
      self.eval()
      with torch.no_grad():
        return self._encode(query_list, is_query=True, is_normalize=is_normalize)


    def encode_documents(
      self,
      document_list: list[str],
      is_normalize=True
    ) -> torch.Tensor:
      self.eval()
      with torch.no_grad():
        return self._encode(document_list, is_normalize=is_normalize)


    def build_index(
      self,
      document_list: list[str],
      batch_size=64,
      device: Literal['cuda', 'cpu']='cuda'
    ):
      document_embedding_list = []

      for index in range(0, len(document_list), batch_size):
        document_batch = document_list[index:index+batch_size]
        with torch.no_grad():
          # TODO: Check faiss, maybe it can work with Tensor and quicker
          document_embedding_bacth = self._encode(
            document_batch,
            device=device,
            is_normalize=True
          ).cpu().numpy().astype('float32')
        document_embedding_list.extend(document_embedding_bacth)
        del document_embedding_bacth
        gc.collect()
        if device == 'cuda':
            torch.cuda.empty_cache()

      document_embedding_list = np.vstack(document_embedding_list)
      nlist = 100
      quantizer = faiss.IndexFlatIP(batch_size)
      retriever_index = faiss.IndexIVFFlat(quantizer, batch_size, nlist, faiss.METRIC_INNER_PRODUCT)

      retriever_index.train(document_embedding_list)
      retriever_index.add(document_embedding_list)
      self.document_list = document_list
      self.retriever_index = retriever_index


    def search(
      self,
      query: str,
      top_k=10
    ) -> list[str]:
      assert self.index is not None, "Firstly call build_index()"

      with torch.no_grad():
        query_embedding = self._encode(
          [query],
          device='cuda',
          is_query=True,
          normalize=True
        ).cpu().numpy().astype('float32')
      _, index_list = self.retriever_index.search(query_embedding, top_k)
      return [self.document_list[index] for index in index_list[0]]


    def save(self, path: str):
      self.encoder.save(path)


    def load(self, path: str):
      self.encoder = SentenceTransformer(path)


In [105]:
def receiver_train_loop(model, dataloader, epochs=3, lr=2e-5, device='cuda'):
    model = model.to(device)
    model.train()

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    for epoch in range(epochs):
        total_loss = 0

        for batch_idx, batch in enumerate(dataloader):
            optimizer.zero_grad()

            loss = model(
                batch['queries'],
                batch['positives'],
                batch['hard_negatives']
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()

            total_loss += loss.item()

            if batch_idx % 10 == 0:
                print(f"Epoch {epoch+1} | Batch {batch_idx} | Loss: {loss.item():.4f}")

        print(f"Epoch {epoch+1} | Avg Loss: {total_loss / len(dataloader):.4f}")

    return model

In [96]:
class TripletDataset(Dataset):
    """
    Датасет для триплетов (query, positive, hard_negative)
    """
    def __init__(self, triplet_list):
        """
        triplet_list: list of (query, positive, hard_negative)
        """
        self.triplets = triplet_list

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        query, positive, hard_negative = self.triplets[idx]
        return {
            'query': query,
            'positive': positive,
            'hard_negative': hard_negative
        }


def collate_triplets(batch):
    """
    Коллайдер для батча триплетов
    """
    queries = [item['query'] for item in batch]
    positives = [item['positive'] for item in batch]
    hard_negatives = [item['hard_negative'] for item in batch]

    return {
        'queries': queries,
        'positives': positives,
        'hard_negatives': hard_negatives
    }

In [106]:
# Создаем датасет и даталоадер
dataset = TripletDataset(triplet_list)
dataloader = DataLoader(
  dataset,
  batch_size=30,
  shuffle=True,
  collate_fn=collate_triplets,
  num_workers=0
)

# Создаем модель
model = PovaryoshkaRetriever(
  encoder_name="deepvk/USER2-small",
  temperature=0.05
)

# Проверяем устройство
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Используем устройство: {device}")

# Обучаем
trained_model = receiver_train_loop(
  model,
  dataloader,
  epochs=20,
  lr=2e-5,
  device=device
)

# Сохраняем
trained_model.save("./povaryoshka_retriever")
print("Модель сохранена!")

Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

Используем устройство: cuda
Epoch 1 | Batch 0 | Loss: 1.6140
Epoch 1 | Avg Loss: 1.6140
Epoch 2 | Batch 0 | Loss: 1.2580
Epoch 2 | Avg Loss: 1.2580
Epoch 3 | Batch 0 | Loss: 0.9455
Epoch 3 | Avg Loss: 0.9455
Epoch 4 | Batch 0 | Loss: 0.6881
Epoch 4 | Avg Loss: 0.6881
Epoch 5 | Batch 0 | Loss: 0.4879
Epoch 5 | Avg Loss: 0.4879
Epoch 6 | Batch 0 | Loss: 0.3377
Epoch 6 | Avg Loss: 0.3377
Epoch 7 | Batch 0 | Loss: 0.2240
Epoch 7 | Avg Loss: 0.2240
Epoch 8 | Batch 0 | Loss: 0.1356
Epoch 8 | Avg Loss: 0.1356
Epoch 9 | Batch 0 | Loss: 0.0707
Epoch 9 | Avg Loss: 0.0707
Epoch 10 | Batch 0 | Loss: 0.0326
Epoch 10 | Avg Loss: 0.0326
Epoch 11 | Batch 0 | Loss: 0.0146
Epoch 11 | Avg Loss: 0.0146
Epoch 12 | Batch 0 | Loss: 0.0070
Epoch 12 | Avg Loss: 0.0070
Epoch 13 | Batch 0 | Loss: 0.0036
Epoch 13 | Avg Loss: 0.0036
Epoch 14 | Batch 0 | Loss: 0.0021
Epoch 14 | Avg Loss: 0.0021
Epoch 15 | Batch 0 | Loss: 0.0012
Epoch 15 | Avg Loss: 0.0012
Epoch 16 | Batch 0 | Loss: 0.0008
Epoch 16 | Avg Loss: 0.000

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Модель сохранена!


In [38]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    dtype=torch.float16
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [110]:
# def generate_answer(prompt):
#   inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

#   # Генерация текста
#   with torch.no_grad():  # отключаем градиенты для inference
#     outputs = model.generate(
#       **inputs,
#       max_new_tokens=100,  # сколько токенов сгенерировать
#       do_sample=True,      # включаем сэмплинг для разнообразия
#       temperature=0.7,     # "творчество" модели
#       top_p=0.9            # nucleus sampling
#   )

#   # Превращаем токены обратно в текст
#   generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

#   return generated_text

In [108]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

def generate_until_eos(prompt, max_total_tokens=1024, temperature=0.7):
    """
    Генерация текста токен за токеном до конца (EOS) или пока не достигнем max_total_tokens.
    """
    model.eval()
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    generated_ids = input_ids

    with torch.no_grad():
        for _ in range(max_total_tokens):
            # Берём последние N токенов как context (N = context_length модели)
            output = model(generated_ids)
            logits = output.logits

            # Берём последний токен
            next_token_logits = logits[:, -1, :] / temperature
            probs = torch.softmax(next_token_logits, dim=-1)

            # Сэмплируем следующий токен
            next_token = torch.multinomial(probs, num_samples=1)
            generated_ids = torch.cat([generated_ids, next_token], dim=1)

            # Если токен EOS → прекращаем генерацию
            if next_token.item() == tokenizer.eos_token_id:
                print(next_token.item())
                break

    return tokenizer.decode(generated_ids[0], skip_special_tokens=True)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [111]:
prompt = "Explain how to cook pasta"

print(generate_until_eos(prompt))

Explain how to cook pasta to make it al dente, and provide tips for achieving the perfect texture and flavor.


In [102]:
def rag_answer(query, retriever):

    retrieved_docs = retriever.search(query)

    context = "\n".join(retrieved_docs)

    prompt = f"""
Контекст:
{context}

Вопрос:
{query}
"""

    return generate_until_eos(prompt)

texts = [chunk['text'] for chunk in chunks]
trained_model.build_index(texts[:1000])

In [76]:
print(rag_answer("Как приготовить борщ?", trained_model))


Контекст:
Инструкции для рецепта "":
1) Сварить из костей наваристый бульон, отдельно до готовности – перловку.

Инструкции для рецепта "":
1) Налить в кастрюлю воду для супа, довести до кипения, подсолить, добавить морковь и картофель, нарезанные небольшими кубиками, варить до мягкости.

Инструкции для рецепта "":
3) Перед подачей татарский суп с мясом сдобрить большим количеством рубленой петрушки и заправить измельченным чесноком.


Вопрос:
Как приготовить борщ?

Ответ:
1) Лист печени куба (кожа) вручную разрезать вдоль гребней.

2) В очень холодном озёре поместить лист печени, листья и стручки ореха, слить в какой-то чемодан смотровую стяжку, заморозить и высушить в течение 1-2 часов.

3) В кастрюлю воды вкушить целый лист печени, слишить из чемодана и перемешать с орехами и крушинами.

4) Дать орехи и крушины в кипяницу, вкушить и оставаясь в кипящем орехе и крушине, добавить небольшое количество масла и тушать до получения мягкой творожки.

5) Перемешать в кастрюлю воду для супа

In [21]:
import torch

# =========================
# 🧠 RAG система
# =========================
class RAGSystem:
    def __init__(self, retriever, generator_fn):
        self.retriever = retriever
        self.generate = generator_fn

    def answer(self, query, top_k=3):
        # 1. Достаем релевантные документы
        docs = self.retriever.search(query, top_k=top_k)

        # 2. Собираем контекст
        context = "\n".join(docs)

        # 3. Формируем prompt
        prompt = f"""
You are a helpful cooking assistant.

Use the context to answer the question.

Context:
{context}

Question:
{query}

Answer:
"""

        # 4. Генерация
        return self.generate(prompt)


# =========================
# 🖥️ CLI интерфейс
# =========================
def chat_loop(rag_system):
    print("🔥 RAG готов! Введи вопрос (exit чтобы выйти)\n")

    while True:
        query = input("👤 Ты: ")

        if query.lower() in ["exit", "quit"]:
            print("Пока 👋")
            break

        answer = rag_system.answer(query)

        print("\n🤖 Ответ:")
        print(answer)
        print("\n" + "="*50 + "\n")

In [22]:
# 1. создаешь retriever
# retriever = PovaryoshkaRetriever()

# # 2. индексируешь документы
# docs = [
#     "Boil pasta in salted water for 10 minutes.",
#     "Add olive oil and garlic for better taste.",
#     "Tomato sauce is commonly used for pasta."
# ]

# retriever.build_index(docs)

# 3. создаешь RAG
rag = RAGSystem(trained_model, generate_until_eos)

# 4. запускаешь чат
chat_loop(rag)

🔥 RAG готов! Введи вопрос (exit чтобы выйти)

👤 Ты: Хочу приготовить завтра утром блинчики. Как их приготовить ?

🤖 Ответ:

You are a helpful cooking assistant.

Use the context to answer the question.

Context:
Инструкции для рецепта "":
2) Печь в духовке до затвердевания, снять, остудить. Для приготовления начинки протереть творог через сито, смешать со сливками с сахаром, взбитыми в крепкую пену. В массу добавить цедру лимона или апельсина, ванилин. Наполнить творогом трубочки из блинчиков, полить растопленным на водяной бане шоколадом. Подать сразу.

Инструкции для рецепта "":
1) Выпечь полтора десятка тонких блинчиков. Формочки для трубочек смазать растопленным сливочным маслом, обернуть готовыми блинчиками внахлест. Чтобы блинчики лучше держались в месте соединения, можно смазать их чуть взбитым яйцом или остатками теста. Излишки обрезать по форме конуса.

Инструкции для рецепта "":
2) Выложить тесто в формы для выпечки, на две трети объема. Печь в духовке в течение получаса, гот

KeyboardInterrupt: Interrupted by user

In [ ]:
import torch
from torch.utils.data import Dataset

class RAGDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=1024):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def build_prompt(self, query, docs, answer):
        context = "\n".join(docs)
        prompt = f"""### Вопрос:
{query}

### Контекст:
{context}

### Ответ:
{answer}"""
        return prompt

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        prompt = self.build_prompt(
            item["query"],
            item["docs"],
            item["answer"]
        )

        tokens = self.tokenizer(
            prompt,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        input_ids = tokens["input_ids"].squeeze(0)
        attention_mask = tokens["attention_mask"].squeeze(0)

        # labels = input_ids (но с маской!)
        labels = input_ids.clone()

        # 🔥 маскируем всё ДО ответа
        answer_start = prompt.find("### Ответ:")
        answer_tokens = self.tokenizer(
            prompt[:answer_start],
            return_tensors="pt"
        )["input_ids"].shape[1]

        labels[:answer_tokens] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

In [ ]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    input_ids = [item["input_ids"] for item in batch]
    attention_mask = [item["attention_mask"] for item in batch]
    labels = [item["labels"] for item in batch]

    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=0)
    attention_mask = pad_sequence(attention_mask, batch_first=True, padding_value=0)
    labels = pad_sequence(labels, batch_first=True, padding_value=-100)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [ ]:
from torch.utils.data import DataLoader

dataset = RAGDataset(data, tokenizer)

dataloader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

# 🔥 LoRA
config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none"
)

model = get_peft_model(model, config)
model.train()

In [ ]:
import torch
from torch.optim import AdamW
from tqdm import tqdm

optimizer = AdamW(model.parameters(), lr=2e-5)

device = next(model.parameters()).device

epochs = 3

for epoch in range(epochs):
    total_loss = 0

    for batch in tqdm(dataloader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch}: loss = {total_loss / len(dataloader)}")

In [ ]:
model.eval()

query = "Как сварить борщ?"
docs = ["Свеклу варят...", "Добавляют картошку..."]

prompt = f"""### Вопрос:
{query}

### Контекст:
{chr(10).join(docs)}

### Ответ:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(device)

output = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Teacher model
teacher_model_name = "meta-llama/Llama-2-7b-chat-hf"  # можно GPT-4 через API
teacher_tokenizer = AutoTokenizer.from_pretrained(teacher_model_name)
teacher_model = AutoModelForCausalLM.from_pretrained(
    teacher_model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

generator = pipeline(
    "text-generation",
    model=teacher_model,
    tokenizer=teacher_tokenizer,
    device=0
)

def generate_target(query, docs):
    context = "\n".join(docs)
    prompt = f"""
Ответь на вопрос, используя только контекст ниже.
Если ответа нет в контексте, скажи "не знаю".

Вопрос: {query}

Контекст:
{context}

Ответ:
"""
    output = generator(prompt, max_new_tokens=150, do_sample=False)
    answer = output[0]["generated_text"].split("Ответ:")[-1].strip()
    return answer

In [ ]:
query = "Как сварить борщ?"
docs = ["Свеклу варят...", "Добавляют картошку...", "Борщ готовится..."]

target_answer = generate_target(query, docs)
print(target_answer)